In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
Torch CUDA version: 11.8
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [ ]:
create pairs , prompt response 

In [11]:
from pathlib import Path
import re

DATA_PATH = Path("../data/data_merged_plus_funny_deduped.txt")
raw = DATA_PATH.read_text(encoding="utf-8", errors="ignore")

def parse_pairs(text: str):
    pairs = []
    chunks = text.split("<END>")
    for ch in chunks:
        ch = ch.strip()
        if not ch:
            continue
        if "<U>" not in ch or "<TGT>" not in ch:
            continue
        try:
            after_u = ch.split("<U>", 1)[1]
            prompt = after_u.split("<TGT>", 1)[0].strip()
            resp   = after_u.split("<TGT>", 1)[1].strip()
        except Exception:
            continue

        prompt = re.sub(r"\s+", " ", prompt).strip()
        resp   = re.sub(r"\s+", " ", resp).strip()
        if not prompt or not resp:
            continue

        pairs.append((prompt, resp))
    return pairs

pairs = parse_pairs(raw)
print("Parsed pairs:", len(pairs))
print("Example:", pairs[0])


Parsed pairs: 20211
Example: ("explain backprop: i'm a fast learner (when i feel like it) lol", 'Okay, you’re giving effort-adjacent energy. You’re giving effort-adjacent energy. it’s okay, growth is a thing.')


In [ ]:
train val test

In [12]:
import random
random.seed(42)
random.shuffle(pairs)

n = len(pairs)
n_train = int(n * 0.90)
n_val   = int(n * 0.05)

train_pairs = pairs[:n_train]
val_pairs   = pairs[n_train:n_train+n_val]
test_pairs  = pairs[n_train+n_val:]

print(len(train_pairs), len(val_pairs), len(test_pairs))

OUT_DIR = Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def write_split(pairs, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for p, r in pairs:
            f.write("<S> USER\n")
            f.write("<U> " + p + "\n")
            f.write("<TGT>\n")
            f.write(r + "\n")
            f.write("<END>\n\n")

write_split(train_pairs, OUT_DIR / "train.txt")
write_split(val_pairs,   OUT_DIR / "val.txt")
write_split(test_pairs,  OUT_DIR / "test.txt")

print("Wrote:", OUT_DIR/"train.txt", OUT_DIR/"val.txt", OUT_DIR/"test.txt")


18189 1010 1012
Wrote: ..\data\train.txt ..\data\val.txt ..\data\test.txt


In [ ]:
tokenise

In [22]:
import sentencepiece as spm

corpus_path = OUT_DIR / "spm_corpus.txt"
with open(corpus_path, "w", encoding="utf-8", newline="\n") as f:
    for p, r in pairs:
        f.write(p + "\n")
        f.write(r + "\n")

spm.SentencePieceTrainer.train(
    input=str(corpus_path),
    model_prefix=str(OUT_DIR / "spm_bpe"),
    vocab_size=8000,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    user_defined_symbols=["<S>", "<U>", "<TGT>", "<END>", "USER"],
    byte_fallback=True
)

sp = spm.SentencePieceProcessor()
sp.load(str(OUT_DIR / "spm_bpe.model"))
vocab_size = sp.get_piece_size()

print("Tokenizer model:", OUT_DIR/"spm_bpe.model")
print("Vocab size:", vocab_size)
print("END piece id:", sp.piece_to_id("<END>"))


Tokenizer model: ..\data\spm_bpe.model
Vocab size: 8000
END piece id: 7


In [ ]:
dataloader

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader


PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

MAX_LEN = 256   # start safe for RTX 3070

def read_blocks(path: Path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    blocks = [b.strip() for b in text.split("<END>") if b.strip()]
    out = []
    for b in blocks:
        if "<U>" not in b or "<TGT>" not in b:
            continue
        try:
            after_u = b.split("<U>", 1)[1]
            prompt = after_u.split("<TGT>", 1)[0].strip()
            resp = after_u.split("<TGT>", 1)[1].strip()
        except:
            continue
        prompt = re.sub(r"\s+", " ", prompt).strip()
        resp   = re.sub(r"\s+", " ", resp).strip()
        if prompt and resp:
            out.append((prompt, resp))
    return out

train_pairs2 = read_blocks(OUT_DIR / "train.txt")
val_pairs2   = read_blocks(OUT_DIR / "val.txt")

class ChatDataset(Dataset):
    def __init__(self, pairs, sp):
        self.pairs = pairs
        self.sp = sp

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        p, r = self.pairs[i]

        # Encoder input: prompt + marker
        src_text = "<S> USER <U> " + p + " <TGT>"
        src = [BOS_ID] + self.sp.encode(src_text, out_type=int) + [EOS_ID]

        # Decoder target: response + end marker
        tgt_text = r + " <END>"
        tgt = [BOS_ID] + self.sp.encode(tgt_text, out_type=int) + [EOS_ID]

        src = src[:MAX_LEN]
        tgt = tgt[:MAX_LEN]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

def collate(batch):
    srcs, tgts = zip(*batch)
    src_max = max(x.size(0) for x in srcs)
    tgt_max = max(x.size(0) for x in tgts)

    src_pad = torch.full((len(batch), src_max), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(batch), tgt_max), PAD_ID, dtype=torch.long)

    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :s.size(0)] = s
        tgt_pad[i, :t.size(0)] = t

    return src_pad, tgt_pad

train_ds = ChatDataset(train_pairs2, sp)
val_ds   = ChatDataset(val_pairs2, sp)

# If your notebook is on Windows and DataLoader hangs, set num_workers=0
num_workers = 0
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate,
                          num_workers=num_workers, pin_memory=True)

print("Train/Val sizes:", len(train_ds), len(val_ds))


Train/Val sizes: 18189 1010


In [ ]:
creating the transformer model :
    seq2seq encoder decoder
positional encoding to keep word order

TransformerSeq2Seq(
  (emb): Embedding(8000, 256, padding_idx=0)
  (pos): PositionalEncoding(Dropout p=0.2)
  (encoder): TransformerEncoder(
    4 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(256, 8 heads)
      (linear1): Linear(256 → 1024)
      (linear2): Linear(1024 → 256)
      (norm1): LayerNorm(256)
      (norm2): LayerNorm(256)
      (dropout...): Dropout(p=0.2)
    )
  )
  (decoder): TransformerDecoder(
    4 x TransformerDecoderLayer(
      (self_attn): MultiheadAttention(256, 8 heads)          # masked causal
      (multihead_attn): MultiheadAttention(256, 8 heads)     # cross-attn
      (linear1): Linear(256 → 1024)
      (linear2): Linear(1024 → 256)
      (norm1/norm2/norm3): LayerNorm(256)
      (dropout...): Dropout(p=0.2)
    )
  )
  (lm_head): Linear(256 → 8000, bias=False)   # tied to embedding weights
)


In [24]:
import math
import torch.nn as nn
import torch

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.2, max_len=2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

def make_pad_mask(tokens, pad_id=0):
    return (tokens == pad_id)

def subsequent_mask(t):
    return torch.triu(torch.ones((t, t), dtype=torch.bool), diagonal=1)

class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_enc=4, num_dec=4,
                 dim_ff=1024, dropout=0.2, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_dec)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # ✅ Weight tying: output projection shares weights with token embedding
        self.lm_head.weight = self.emb.weight

    def forward(self, src, tgt_in):
        src_key_padding = make_pad_mask(src, self.pad_id)
        tgt_key_padding = make_pad_mask(tgt_in, self.pad_id)

        src_emb = self.pos(self.emb(src) * math.sqrt(self.d_model))
        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding)

        tgt_emb = self.pos(self.emb(tgt_in) * math.sqrt(self.d_model))
        T = tgt_in.size(1)
        causal = subsequent_mask(T).to(tgt_in.device)

        out = self.decoder(
            tgt_emb, memory,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_key_padding,
            memory_key_padding_mask=src_key_padding
        )
        return self.lm_head(out)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerSeq2Seq(
    vocab_size=vocab_size,
    d_model=256, nhead=8,
    num_enc=4, num_dec=4,
    dim_ff=1024,
    dropout=0.2,
    pad_id=PAD_ID
).to(DEVICE)

sum(p.numel() for p in model.parameters())


9420800

In [ ]:
train
label smoothing to reduce confidence in one token only
ignoring padding
adamw 
weight decay for generalisation

In [25]:
import torch.nn.functional as F
from tqdm import tqdm

def label_smoothed_loss(logits, target, eps=0.1):
    # logits: (B,T,V), target: (B,T)
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    smooth = -log_probs.mean(dim=-1)

    mask = (target != PAD_ID)
    nll = (nll * mask).sum() / mask.sum().clamp_min(1)
    smooth = (smooth * mask).sum() / mask.sum().clamp_min(1)
    return (1 - eps) * nll + eps * smooth

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.98), weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

best_val = float("inf")
patience = 3
bad = 0
ckpt_path = OUT_DIR / "best.pt"

for epoch in range(1, 41):
    # ---- train
    model.train()
    train_loss_sum = 0
    train_tok_sum = 0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        src = src.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
            logits = model(src, tgt_in)
            loss = label_smoothed_loss(logits, tgt_out, eps=0.1)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        mask = (tgt_out != PAD_ID)
        toks = mask.sum().item()
        train_loss_sum += loss.item() * toks
        train_tok_sum += toks

    train_loss = train_loss_sum / max(1, train_tok_sum)

    # ---- val
    model.eval()
    val_loss_sum = 0
    val_tok_sum = 0

    with torch.no_grad():
        for src, tgt in tqdm(val_loader, desc=f"Epoch {epoch} val"):
            src = src.to(DEVICE, non_blocking=True)
            tgt = tgt.to(DEVICE, non_blocking=True)

            tgt_in = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                logits = model(src, tgt_in)
                loss = label_smoothed_loss(logits, tgt_out, eps=0.1)

            mask = (tgt_out != PAD_ID)
            toks = mask.sum().item()
            val_loss_sum += loss.item() * toks
            val_tok_sum += toks

    val_loss = val_loss_sum / max(1, val_tok_sum)

    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        bad = 0
        torch.save({"model": model.state_dict()}, ckpt_path)
        print("✅ Saved best:", ckpt_path)
    else:
        bad += 1
        if bad >= patience:
            print("⏹️ Early stop.")
            break


C:\Users\fessu\AppData\Local\Temp\ipykernel_9612\1878810328.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
Epoch 1 train:   0%|                                                                           | 0/569 [00:00<?, ?it/s]C:\Users\fessu\AppData\Local\Temp\ipykernel_9612\1878810328.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
Epoch 1 val:   0%|                                                                              | 0/32 [00:00<?, ?it/s]C:\Users\fessu\AppData\Local\Temp\ipykernel_9612\1878810328.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
Epoch 1 v

Epoch 01 | train 16.1952 | val 8.1306
✅ Saved best: ..\data\best.pt


Epoch 2 val: 100%|█████████████████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 23.67it/s]


Epoch 02 | train 7.2638 | val 5.4223
✅ Saved best: ..\data\best.pt


Epoch 3 val: 100%|█████████████████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 22.91it/s]


...

Epoch 38 | train 1.9071 | val 1.8235
✅ Saved best: ..\data\best.pt


Epoch 39 val: 100%|████████████████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 20.77it/s]


Epoch 39 | train 1.9053 | val 1.8217
✅ Saved best: ..\data\best.pt


Epoch 40 val: 100%|████████████████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 20.15it/s]

Epoch 40 | train 1.8986 | val 1.8209
✅ Saved best: ..\data\best.pt


In [ ]:
removing garbage( post processing )
taking top_k
improving first 5 tokens
repetition penalty

In [29]:
import torch

END_ID = sp.piece_to_id("<END>")
UNK_ID = sp.unk_id()
PAD_ID = 0

def top_k_top_p_sample(logits, top_k=50, top_p=0.90, temperature=0.85):
    logits = logits / max(1e-6, temperature)

    # top-k filter
    if top_k is not None and top_k > 0:
        v, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
        cutoff = v[-1]
        logits = torch.where(logits < cutoff, torch.tensor(-1e9, device=logits.device), logits)

    probs = torch.softmax(logits, dim=-1)

    # top-p filter
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    cutoff = cum > top_p
    cutoff[..., 1:] = cutoff[..., :-1].clone()
    cutoff[..., 0] = False
    sorted_probs[cutoff] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

    idx = torch.multinomial(sorted_probs, 1)
    next_id = sorted_idx.gather(-1, idx).squeeze(-1)
    return next_id

def apply_repetition_penalty(logits, generated_ids, penalty=1.12):
    if penalty <= 1.0:
        return logits
    for tid in set(generated_ids):
        if 0 <= tid < logits.numel():
            logits[tid] = logits[tid] / penalty if logits[tid] > 0 else logits[tid] * penalty
    return logits

@torch.no_grad()
def generate(prompt, max_new_tokens=100, top_p=0.90, temperature=0.85,
             repetition_penalty=1.12, no_repeat_ngram_size=3):

    model.eval()

    src_text = "<S> USER <U> " + prompt + " <TGT>"
    src_ids = [BOS_ID] + sp.encode(src_text, out_type=int) + [EOS_ID]
    src = torch.tensor(src_ids[:MAX_LEN], dtype=torch.long, device=DEVICE).unsqueeze(0)

    tgt = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)
    gen_ids = []
    ngram_bans = {}

    for step in range(max_new_tokens):
        logits = model(src, tgt)
        next_logits = logits[0, -1].clone()

        # ban UNK/PAD always
        next_logits[UNK_ID] = -1e9
        next_logits[PAD_ID] = -1e9

        # extra safety for first few tokens: be less random
        if step < 5:
            top_k = 30
            local_temp = 0.75
            local_top_p = 0.85
        else:
            top_k = 50
            local_temp = temperature
            local_top_p = top_p

        next_logits = apply_repetition_penalty(next_logits, gen_ids, penalty=repetition_penalty)

        if no_repeat_ngram_size > 1 and len(gen_ids) >= no_repeat_ngram_size - 1:
            prefix = tuple(gen_ids[-(no_repeat_ngram_size - 1):])
            banned = ngram_bans.get(prefix, None)
            if banned:
                next_logits[list(banned)] = -1e9

        next_id = int(top_k_top_p_sample(next_logits, top_k=top_k, top_p=local_top_p, temperature=local_temp).item())

        if no_repeat_ngram_size > 1 and len(gen_ids) >= no_repeat_ngram_size - 1:
            prefix = tuple(gen_ids[-(no_repeat_ngram_size - 1):])
            ngram_bans.setdefault(prefix, set()).add(next_id)

        gen_ids.append(next_id)
        tgt = torch.cat([tgt, torch.tensor([[next_id]], device=DEVICE)], dim=1)

        if next_id == EOS_ID or next_id == END_ID:
            break
        if tgt.size(1) >= MAX_LEN:
            break

    text = sp.decode(gen_ids)
    text = text.replace("<END>", "").strip()
    return text


# Test again
prompts = [
    # greetings / small talk
    "yo",
    "hey, you there?",
    "what's up",
    "how's it going",
    "tell me something funny",

    # boredom / motivation
    "im bored at home",
    "i have zero motivation today",
    "convince me to stop procrastinating",
    "i keep quitting when it gets hard",

    # feelings / mental vibe (still light)
    "i feel anxious for no reason",
    "i feel sad and i don't know why",
    "i'm stressed about everything",
    "i feel lonely lately",
    "i can't sleep and my brain won't shut up",

    # relationships / social
    "my friend left me on read, what do i do",
    "i think my crush doesn't like me back",
    "i got into an argument with my mom",
    "how do i say sorry without sounding fake",
    "my friend keeps copying me and it's annoying",

    # school/work life
    "i have an exam tomorrow and i didn't study",
    "i hate my job but i need money",
    "my boss is annoying but i can't say it",
    "i have a presentation and i'm nervous",
    "how do i focus for more than 10 minutes",

    # practical / everyday questions
    "what should i eat for dinner",
    "how do i stop my phone addiction",
    "give me a quick workout i can do at home",
    "my laptop is slow, what do i try first",
    "how do i start journaling",

    # opinion / debate style
    "is pineapple on pizza actually good",
    "cats or dogs",
    "be honest: am i the problem",
    "roast me",
    "compliment me but make it sarcastic",

    # “follow-up / multi-turn” style
    "i failed my exam",
    "ok but what if i fail again",
    "i'm trying but nothing changes",
    "give me a plan for tomorrow",
    "make it simpler",
]

for i, p in enumerate(prompts, 1):
    print(f"\n--- Example {i} ---")
    print("USER:", p)
    print("BOT :", generate(p))



--- Example 1 ---
USER: yo
BOT : I’m begging you. your idea should come with a warning label. Your brain just took a smoke break.

--- Example 2 ---
USER: hey, you there?
BOT : Respectfully, your logic is buffering. Your plan is held together by vibes and duct tape. take a lap and come back with a better plan.

--- Example 3 ---
USER: what's up
BOT : I’m crying, your explanation makes less sense the longer it goes. You’re allergic to accountability.

--- Example 4 ---
USER: how's it going
BOT : Hello! How can I help you today?

--- Example 5 ---
USER: tell me something funny
BOT : Bad joke: I’m a joke about anti‐gravity. It’s impossible to put down.

--- Example 6 ---
USER: im bored at home
BOT : I’m crying, your logic is buffering. Your plan is held together by vibes and duct tape.

--- Example 7 ---
USER: i have zero motivation today
BOT : i'm crying, your plan is held together by vibes and duct tape. Your self-awareness just went offline.

--- Example 8 ---
USER: convince me to sto

In [ ]:
making chatbot to test ( the chatbot is annoying btw

In [36]:
# Simple interactive chat loop (type 'exit' to quit)

import torch
from pathlib import Path

# (Optional) load best checkpoint if not already loaded
ckpt_path = Path("../data/best.pt")
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print("✅ Loaded model from", ckpt_path)

print("\nChat is ready. Type 'exit' / 'quit' to stop.\n")

while True:
    user = input("You: ").strip()
    if not user:
        continue
    if user.lower() in {"exit", "quit", "q"}:
        print("Bot: aight. try not to miss me too much.")
        break

    bot = generate(user, max_new_tokens=120, top_p=0.90, temperature=0.85)
    print("Bot:", bot)


✅ Loaded model from ..\data\best.pt

Chat is ready. Type 'exit' / 'quit' to stop.



You:  are you good ?


Bot: Not you thinking your self-awareness just went offline. Your logic is buffering. take a lap and come back with a better plan.


You:  exist


Bot: I’m crying, your explanation makes less sense the longer it goes. Your plan is held together by vibes and duct tape.


You:  exit


Bot: aight. try not to miss me too much.


In [37]:
# Save EVERYTHING needed to run the chatbot anywhere into ../model
from pathlib import Path
import json, shutil, sys, subprocess

MODEL_DIR = Path("../model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1) Save weights (prefer best.pt if it exists) ----
data_dir = Path("../data")
best_ckpt = data_dir / "best.pt"

weights_out = MODEL_DIR / "weights.pt"
if best_ckpt.exists():
    shutil.copy2(best_ckpt, weights_out)
    print(f"✅ Copied checkpoint -> {weights_out}")
else:
    # fall back to in-memory model
    assert "model" in globals(), "No ../data/best.pt found and `model` is not in memory."
    import torch
    torch.save({"model": model.state_dict()}, weights_out)
    print(f"✅ Saved in-memory model -> {weights_out}")

# ---- 2) Save SentencePiece tokenizer files ----
spm_model = data_dir / "spm_bpe.model"
spm_vocab = data_dir / "spm_bpe.vocab"

assert spm_model.exists(), f"Missing {spm_model} (run SentencePiece training cell first)."
shutil.copy2(spm_model, MODEL_DIR / "spm_bpe.model")
if spm_vocab.exists():
    shutil.copy2(spm_vocab, MODEL_DIR / "spm_bpe.vocab")

print("✅ Tokenizer saved")

# ---- 3) Save config (hyperparams + special ids) ----
# Try to infer from existing globals; fall back to safe defaults.
cfg = {
    "vocab_size": int(getattr(getattr(model, "emb", None), "num_embeddings", 8000)) if "model" in globals() else 8000,
    "d_model": int(getattr(getattr(model, "emb", None), "embedding_dim", 256)) if "model" in globals() else 256,
    "nhead": int(getattr(getattr(model, "enc_layer", None), "self_attn", None).num_heads) if "model" in globals() and hasattr(model, "enc_layer") else 8,
    "num_layers": int(getattr(model, "num_layers", 4)) if "model" in globals() else 4,
    "dim_ff": int(getattr(getattr(getattr(model, "enc_layer", None), "linear1", None), "out_features", 1024)) if "model" in globals() and hasattr(model, "enc_layer") else 1024,
    "dropout": float(getattr(getattr(getattr(model, "pos", None), "dropout", None), "p", 0.2)) if "model" in globals() else 0.2,
    "max_len": int(globals().get("MAX_LEN", 256)),
    "pad_id": int(globals().get("PAD_ID", 0)),
    "bos_id": int(globals().get("BOS_ID", 2)),
    "eos_id": int(globals().get("EOS_ID", 3)),
    "end_id": int(globals().get("END_ID", -1)),  # may be -1 if you didn't add <END> to spm
}

with open(MODEL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("✅ config.json saved")

# ---- 4) Write a standalone inference script (python infer.py) ----
infer_py = r'''
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sentencepiece as spm

HERE = Path(__file__).resolve().parent
cfg = json.loads((HERE / "config.json").read_text(encoding="utf-8"))

PAD_ID = cfg["pad_id"]
BOS_ID = cfg["bos_id"]
EOS_ID = cfg["eos_id"]
END_ID = cfg.get("end_id", -1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.2, max_len=2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_ff=1024, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout, batch_first=True)
        dec_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout, batch_first=True)

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)

        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None):
        # src,tgt: (B,T)
        src_emb = self.pos(self.emb(src))
        tgt_emb = self.pos(self.emb(tgt))

        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

        # causal mask for decoder
        T = tgt.size(1)
        causal = torch.triu(torch.ones(T, T, device=tgt.device), diagonal=1).bool()

        dec = self.decoder(
            tgt_emb, memory,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.out(dec)

def top_k_top_p_sample(logits, top_k=50, top_p=0.90, temperature=0.85):
    logits = logits / max(1e-6, temperature)

    if top_k and top_k > 0:
        v, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
        cutoff = v[-1]
        logits = torch.where(logits < cutoff, torch.full_like(logits, -1e10), logits)

    probs = torch.softmax(logits, dim=-1)
    if top_p and 0 < top_p < 1:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=-1)
        mask = cumsum > top_p
        mask[..., 0] = False
        sorted_probs = torch.where(mask, torch.zeros_like(sorted_probs), sorted_probs)
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        next_in_sorted = torch.multinomial(sorted_probs, 1).item()
        return sorted_idx[next_in_sorted].item()

    return torch.multinomial(probs, 1).item()

@torch.no_grad()
def generate(model, sp, prompt, max_new_tokens=128, top_k=50, top_p=0.90, temperature=0.85, device="cpu"):
    model.eval()
    # encode prompt
    src_ids = sp.encode(prompt, out_type=int)
    src = torch.tensor([src_ids[:cfg["max_len"]]], device=device, dtype=torch.long)

    # masks
    src_pad = (src == PAD_ID)

    # start decoder with BOS
    ys = torch.tensor([[BOS_ID]], device=device, dtype=torch.long)

    for _ in range(max_new_tokens):
        tgt_pad = (ys == PAD_ID)
        logits = model(src, ys, src_key_padding_mask=src_pad, tgt_key_padding_mask=tgt_pad)
        next_logits = logits[0, -1]  # (V,)
        nxt = top_k_top_p_sample(next_logits, top_k=top_k, top_p=top_p, temperature=temperature)
        ys = torch.cat([ys, torch.tensor([[nxt]], device=device)], dim=1)

        if nxt == EOS_ID:
            break
        if END_ID != -1 and nxt == END_ID:
            break

    out_ids = ys[0].tolist()
    # drop BOS
    if out_ids and out_ids[0] == BOS_ID:
        out_ids = out_ids[1:]
    # stop at EOS/END
    stop_ids = {EOS_ID}
    if END_ID != -1:
        stop_ids.add(END_ID)
    cut = len(out_ids)
    for i,t in enumerate(out_ids):
        if t in stop_ids:
            cut = i
            break
    out_ids = out_ids[:cut]
    return sp.decode(out_ids)

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sp = spm.SentencePieceProcessor()
    sp.load(str(HERE / "spm_bpe.model"))

    model = TransformerSeq2Seq(
        vocab_size=cfg["vocab_size"],
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        dim_ff=cfg["dim_ff"],
        dropout=cfg["dropout"],
    ).to(device)

    ckpt = torch.load(HERE / "weights.pt", map_location=device)
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(sd, strict=True)

    print("Chat ready. Type 'exit' to quit.")
    while True:
        msg = input("\nYou: ").strip()
        if msg.lower() in {"exit", "quit"}:
            break
        print("Bot:", generate(model, sp, msg, device=device))

if __name__ == "__main__":
    main()
'''
(MODEL_DIR / "infer.py").write_text(infer_py, encoding="utf-8")
print("✅ infer.py written")

# ---- 5) Minimal requirements ----
req = "\n".join([
    "torch",
    "sentencepiece",
])
(MODEL_DIR / "requirements.txt").write_text(req + "\n", encoding="utf-8")
print("✅ requirements.txt written")

print(f"\n🎉 Done. Your portable package is in: {MODEL_DIR.resolve()}")
print("Run anywhere:")
print("  pip install -r requirements.txt")
print("  python infer.py")


✅ Copied checkpoint -> ..\model\weights.pt
✅ Tokenizer saved
✅ config.json saved
✅ infer.py written
✅ requirements.txt written

🎉 Done. Your portable package is in: C:\Users\fessu\OneDrive\Desktop\personal_work\transformer_from_scratch\model
Run anywhere:
  pip install -r requirements.txt
  python infer.py
